In [1]:
import numpy as np
from einops import rearrange
from unsupervised_retrieval_metrics import UnsupervisedRetrievalEvaluator

def generate_mock_proprioceptive_data(
    num_sequences=100,
    seq_len=100,
    num_features=14,
    tasks=6,
):
    """
    Generate mock proprioceptive data.
    Loop hierarchy: for each task, create num_sequence sequences of length seq_len, with num_features features.
    Output shape: [num_sequences, seq_len, num_features, tasks]
    Each feature is always non-degenerate, no zero rows.
    """
    rng = np.random.default_rng(42)
    data = np.zeros((num_sequences, seq_len, num_features, tasks), dtype=np.float32)
    t_axis = np.linspace(0, 4 * np.pi, seq_len)

    for task_idx in range(tasks):
        for seq_idx in range(num_sequences):
            for feat_idx in range(num_features):
                freq = rng.uniform(0.2, 1.2) + 0.1 * task_idx
                phase = rng.uniform(0, 2 * np.pi) + 0.2 * task_idx
                amplitude = rng.uniform(0.8, 1.2) + 0.05 * task_idx
                drift = rng.uniform(-0.02, 0.02) + 0.002 * task_idx
                baseline = rng.uniform(-0.5, 0.5) + 0.05 * task_idx

                base_signal = amplitude * np.sin(freq * t_axis + phase) + drift * t_axis + baseline
                base_signal += rng.normal(0, 0.03, size=seq_len)  # Small persistent noise

                data[seq_idx, :, feat_idx, task_idx] = base_signal

    for task_idx in range(tasks):
        for feat_idx in range(num_features):
            for seq_idx in range(num_sequences):
                assert not np.allclose(data[seq_idx, :, feat_idx, task_idx], data[seq_idx, 0, feat_idx, task_idx])

    return data[:, :, :, :(tasks-1)], data[:, :, :, tasks-1]


import numpy as np

# -----------------------------
# STRAP Core Components
# -----------------------------

def zscore(ts, eps=1e-6):
    """
    ts: (T, F)
    """
    mean = ts.mean(axis=0, keepdims=True)
    std = ts.std(axis=0, keepdims=True)
    return (ts - mean) / (std + eps)


def multivariate_dtw(x, y):
    """
    Multivariate DTW (variable-length supported)

    x: (T_x, F)
    y: (T_y, F)
    returns: scalar distance
    """
    T_x, T_y = x.shape[0], y.shape[0]
    cost = np.full((T_x + 1, T_y + 1), np.inf, dtype=np.float32)
    cost[0, 0] = 0.0

    for i in range(1, T_x + 1):
        for j in range(1, T_y + 1):
            dist = np.linalg.norm(x[i - 1] - y[j - 1])
            cost[i, j] = dist + min(
                cost[i - 1, j],
                cost[i, j - 1],
                cost[i - 1, j - 1],
            )

    return cost[T_x, T_y]


# -----------------------------
# STRAP Retrieval
# -----------------------------

class STRAPRetriever:
    """
    Shape-based Task-Agnostic Proprioceptive retrieval

    Retrieves K trajectories from a general pool based on
    DTW distance to a reference set.
    """

    def __init__(self, general_data):
        """
        general_data: list or array of shape [(T_i, F)] or (N, T_i, F)
        """
        self.general_raw = general_data
        self.general = [zscore(x) for x in general_data]

    def retrieve(self, reference_data, num_retrievals=100):
        """
        reference_data: iterable of (T_ref, F)
        num_retrievals: number of trajectories to retrieve

        Returns:
          retrieved_data: list of arrays [(T_g, F)]
          distances: (K,)
        """
        reference = [zscore(x) for x in reference_data]

        print("Computing DTW distances...")

        all_distances = []
        all_indices = []

        for ref in reference:
            dists = np.array(
                [multivariate_dtw(ref, g) for g in self.general],
                dtype=np.float32,
            )
            all_distances.append(dists)
            all_indices.append(np.arange(len(self.general)))

        # Concatenate across references (task-agnostic retrieval)
        all_distances = np.concatenate(all_distances)
        all_indices = np.concatenate(all_indices)

        # Top-K smallest distances globally
        topk_idx = np.argsort(all_distances)[:num_retrievals]

        retrieved_data = [self.general_raw[all_indices[i]] for i in topk_idx]
        distances = all_distances[topk_idx]

        print(f"Retrieved {len(retrieved_data)} trajectories.")

        retrieved_data = np.array(retrieved_data)
        distances = np.array(distances)

        return retrieved_data, distances


In [2]:
num_samples, seq_len, num_features, num_tasks = 50, 25, 7, 6
general_data, reference_data = generate_mock_proprioceptive_data(num_samples, seq_len, num_features, num_tasks)
print("General data shape:", general_data.shape)
print("Reference data shape:", reference_data.shape)

General data shape: (50, 25, 7, 5)
Reference data shape: (50, 25, 7)


In [ ]:
general_data = rearrange(
    general_data,
    's t f k -> (s k) t f'
)
np.random.shuffle(general_data)
print("Shuffled general data shape:", general_data.shape)

print("General pool shape:", general_data.shape)
print("Reference shape:", reference_data.shape)

retriever = STRAPRetriever(general_data)
retrieved_data, indices = retriever.retrieve(reference_data, num_retrievals=100)

print("Retrieved indices shape:", indices.shape)
print("Retrieved distances shape:", retrieved_data.shape)

Shuffled general data shape: (250, 25, 7)
General pool shape: (250, 25, 7)
Reference shape: (50, 25, 7)
Computing DTW distances...
Retrieved 50 trajectories.
Retrieved indices shape: (50,)
Retrieved distances shape: (50, 25, 7)


In [4]:
evaluator = UnsupervisedRetrievalEvaluator(
    retrieved_data=retrieved_data,
    reference_data=reference_data
)
results = evaluator.full_analysis()
print("Average-feature evaluation complete.")

Average-feature evaluation complete.
